# Copernicus Sentinel-2 Exploration

This notebook showcases how to use the `geospatial_tools` library to search for and download Sentinel-2 products from the **Copernicus Data Space Ecosystem (CDSE)** STAC catalog. 

We will demonstrate:
1. Initializing the `StacSearch` client for Copernicus.
2. Performing a spatial and temporal search.
3. Inspecting STAC items and assets.
4. Downloading data using the **S3 Protocol** for optimized access.

## Prerequisites
1. Create a Copernicus Account here : []()
2. Register an S3 access key here : [https://documentation.dataspace.copernicus.eu/APIs/S3.html](https://documentation.dataspace.copernicus.eu/APIs/S3.html)
3. Create your own .env file from the .env.example file with your copernicus credentials

In [9]:

import leafmap
import shutil
from geospatial_tools.stac.core import StacSearch, COPERNICUS
from geospatial_tools.stac.copernicus import CopernicusS2Collection, CopernicusS2Band, CopernicusS2Property
from geospatial_tools import TESTS_DIR

In [10]:
# Folder setup
TEST_TMP_DIR = TESTS_DIR / "test_notebooks/tmp_copernicus_sentinel2"
TEST_TMP_DIR.mkdir(exist_ok=True)

## 1. Define Search Parameters

We'll search for Sentinel-2 Level-2A data over Rome, Italy, for a specific period in 2024.

In [11]:
# Bounding box for Rome [min_lon, min_lat, max_lon, max_lat]
bbox = [12.4, 41.8, 12.5, 41.9]
date_range = "2024-07-01/2024-07-31"
collections = [CopernicusS2Collection.L2A]

print(f"Searching in {bbox} for period {date_range}...")

Searching in [12.4, 41.8, 12.5, 41.9] for period 2024-07-01/2024-07-31...


## 2. Initialize StacSearch and Query

Initializing `StacSearch` with `catalog_name=COPERNICUS` automatically configures the S3 client using the credentials found in your `.env` file.

In [12]:
stac_search = StacSearch(catalog_name=COPERNICUS)

results = stac_search.search(
    bbox=bbox, 
    date_range=date_range, 
    collections=collections,
    max_items=1
)

print(f"Found {len(results)} items.")

[2026-04-20 13:12:56] INFO       [MainThread][geospatial_tools.stac.utils] Creating S3 client with endpoint: [https://eodata.dataspace.copernicus.eu]
[2026-04-20 13:12:56] INFO       [MainThread][geospatial_tools.stac.core] Initiating STAC API search
Found 1 items.


## 3. Inspect Results

Let's look at the first item and its available assets.

In [13]:
if results:
    item = results[0]
    print(f"Item ID: {item.id}")
    print(f"Cloud Cover: {item.properties.get(CopernicusS2Property.CLOUD_COVER)}% ")
    
    # List a few bands/assets
    available_assets = list(item.assets.keys())
    print(f"First 10 assets: {available_assets[:10]}")

Item ID: S2B_MSIL2A_20240728T095549_N0511_R122_T33TTG_20240728T114034
Cloud Cover: 0.0% 
First 10 assets: ['AOT_10m', 'AOT_20m', 'AOT_60m', 'B01_20m', 'B01_60m', 'B02_10m', 'B02_20m', 'B02_60m', 'B03_10m', 'B03_20m']


We can visualize the search area and the footprint of the first result using `leafmap`.

## 4. Download Assets using S3

Now we download the True Color Image (TCI) and the NIR band (B08) using the optimized S3 protocol.

In [14]:
download_dir = TEST_TMP_DIR / "sentinel-2" / "copernicus_exploration"
download_dir.mkdir(parents=True, exist_ok=True)

bands = [
    CopernicusS2Band.TCI,
    CopernicusS2Band.B08
]

print(f"Downloading bands {bands} to {download_dir} via S3...")

# download_search_results will handle the dispatcher logic automatically
downloaded_assets = stac_search.download_search_results(
    bands=bands, 
    base_directory=download_dir
)

for asset in downloaded_assets:
    asset.show_asset_items()

[2026-04-20 13:12:57] INFO       [MainThread][geospatial_tools.stac.core] Downloading [S2B_MSIL2A_20240728T095549_N0511_R122_T33TTG_20240728T114034] ...
[2026-04-20 13:12:57] INFO       [MainThread][geospatial_tools.stac.core] Retrieving Copernicus credentials...
[2026-04-20 13:12:57] INFO       [MainThread][geospatial_tools.stac.core] Successfully retrieved Copernicus credentials.
[2026-04-20 13:12:58] INFO       [MainThread][geospatial_tools.stac.core] Successfully obtained Copernicus access token.
[2026-04-20 13:12:58] INFO       [MainThread][geospatial_tools.stac.core] Downloading TCI_10m from s3://eodata/Sentinel-2/MSI/L2A/2024/07/28/S2B_MSIL2A_20240728T095549_N0511_R122_T33TTG_20240728T114034.SAFE/GRANULE/L2A_T33TTG_A038615_20240728T095731/IMG_DATA/R10m/T33TTG_20240728T095549_TCI_10m.jp2 using method [s3]
[2026-04-20 13:12:58] INFO       [MainThread][geospatial_tools.stac.core] Downloading from S3: bucket=[eodata], key=[Sentinel-2/MSI/L2A/2024/07/28/S2B_MSIL2A_20240728T095549_N05

In [15]:
tci_asset = downloaded_assets[0][CopernicusS2Band.TCI]
m = leafmap.Map()
m.add_raster(source=str(tci_asset.filename))
m

Map(center=[41.9185435, 12.0389935], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title…

## Conclusion

By using the `StacSearch` dispatcher with the `COPERNICUS` catalog, you can seamlessly switch from HTTP downloads to direct S3 bucket access, which is significantly faster and more reliable for large-scale data retrieval.

In [16]:
shutil.rmtree(TEST_TMP_DIR)